# Hidraulična preša — predvidi, izračunaj, provjeri

**Poglavlje U01: fluid kao kontinuum, tlak i Pascalov zakon**

Osnovni scenarij koristi podatke Z2: promjere 28 i 140 mm, silu 180 N i hod 120 mm. Uz račun sile pratit ćemo
pomak, rad i utjecaj mjernih nesigurnosti. Osnovni model pretpostavlja kvazistatički rad, jednake visine klipova i
nestlačivi fluid, krute klipove, zanemarivo trenje i jednak referentni tlak
s vanjske strane oba klipa.


## 1. Predvidi

Prije izvođenja koda zapiši svoje odgovore:

1. Ako se promjer izlaznog klipa udvostruči, koliko se puta mijenja sila $F_2$?
2. Ako mali klip prijeđe $s_1=120$ mm, je li pomak velikog klipa veći ili manji?
3. Koje će mjerenje više utjecati na $F_2$: pogreška od 1 % u promjeru ili 1 % u sili?

4. Može li promjena volumena tekućine od samo 0,04 % ipak promijeniti pomak za više od 1 %?
5. Zašto najmanji ponuđeni promjer pumpe u Z5 nije nužno prihvatljiv?
6. Koji krajevi intervala učinkovitosti određuju konzervativnu odluku u Z6?

Tek nakon toga pokreni osnovni račun.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

def presa(D1_mm, D2_mm, F1_N, s1_mm=120.0):
    # Idealna preša; sve se interne veličine računaju u SI jedinicama.
    D1, D2 = np.asarray([D1_mm, D2_mm], dtype=float) / 1000.0
    A1, A2 = np.pi * D1**2 / 4.0, np.pi * D2**2 / 4.0
    p = F1_N / A1
    F2 = p * A2
    s2_mm = s1_mm * A1 / A2
    return {"A1": A1, "A2": A2, "p": p, "F2": F2,
            "s1_mm": s1_mm, "s2_mm": s2_mm}

osnovno = presa(D1_mm=28.0, D2_mm=140.0, F1_N=180.0, s1_mm=120.0)
for oznaka, vrijednost in osnovno.items():
    print(f"{oznaka:6s} = {vrijednost:.6g}")


## 2. Izračunaj — parametarska osjetljivost

Za idealni model vrijedi $F_2/F_1=(D_2/D_1)^2$, dok je
$s_2/s_1=(D_1/D_2)^2$. Mreža omjera promjera pokazuje kako se dobitak sile
plaća jednakim gubitkom pomaka. To je parametarska analiza, a ne nova formula.


In [ ]:
omjer_D = np.linspace(1.0, 7.0, 121)
dobitak_sile = omjer_D**2
omjer_pomaka = 1.0 / dobitak_sile

fig, ax1 = plt.subplots(figsize=(7.2, 4.2))
ax1.plot(omjer_D, dobitak_sile, color="#1565c0", label=r"$F_2/F_1$")
ax1.set(xlabel=r"omjer promjera $D_2/D_1$", ylabel="dobitak sile")
ax2 = ax1.twinx()
ax2.plot(omjer_D, omjer_pomaka, color="#c62828", label=r"$s_2/s_1$")
ax2.set_ylabel("omjer pomaka")
ax1.grid(ls=":", alpha=0.6)
fig.tight_layout()
plt.show()


## 3. Provjeri — bilance i mjerna nesigurnost

Prva neovisna provjera jest jednak tlak na oba klipa. Druga je jednakost
idealnih radova $F_1s_1=F_2s_2$. Za male, međusobno neovisne standardne
nesigurnosti linearizacija daje

$$
\left(\frac{u_{F_2}}{F_2}\right)^2=
\left(\frac{u_{F_1}}{F_1}\right)^2+
\left(2\frac{u_{D_2}}{D_2}\right)^2+
\left(2\frac{u_{D_1}}{D_1}\right)^2.
$$


In [ ]:
D1, D2, F1 = 28.0, 140.0, 180.0
# Dodatne sintetičke standardne nesigurnosti za ovaj pokus; nisu podatci Z2.
u_D1, u_D2, u_F1 = 0.10, 0.20, 2.0  # mm, mm, N
r = presa(D1, D2, F1)
p_na_izlazu = r["F2"] / r["A2"]
W_ulaz = F1 * (r["s1_mm"] / 1000.0)
W_izlaz = r["F2"] * (r["s2_mm"] / 1000.0)

u_rel = np.sqrt((u_F1/F1)**2 + (2*u_D2/D2)**2 + (2*u_D1/D1)**2)
u_F2 = r["F2"] * u_rel
print(f"F2 = {r['F2']:.1f} ± {u_F2:.1f} N (standardna nesigurnost)")
print(f"idealni rad: ulaz {W_ulaz:.5f} J, izlaz {W_izlaz:.5f} J")

# Neovisne tvrdnje: prijenos tlaka, bilanca rada i granični slučaj.
assert np.isclose(r["p"], p_na_izlazu, rtol=1e-12)
assert np.isclose(W_ulaz, W_izlaz, rtol=1e-12)
assert np.isclose(presa(50, 50, 123)["F2"], 123.0, rtol=1e-12)
print("PASS: tlak, idealni rad i jednaki klipovi daju tri neovisne provjere.")

assert np.isclose(r["F2"], 4500.0, rtol=1e-12)
assert np.isclose(r["s2_mm"], 4.8, rtol=1e-12)


## 4. Z4 — mjerenje geometrije i provjera stlačivosti

Tri sintetička para volumena i pomaka služe za određivanje iste površine.
Odvojeni pokus s blokiranim klipom daje silu. U trećem pokusu **masa cijele
zatvorene tekućine ostaje ista**, uključujući pumpu. Pumpa smanjuje komoru
za 5,00 cm³; tlak raste za 0,40 MPa pri stalnoj temperaturi.

Uz $K=1,00$ GPa i početnih 250 cm³, smanjenje volumena približno je
$V_0\Delta p/K$. Samo ostatak istisnutog volumena daje pomak klipa.
Za konstantan diferencijalni modul vrijedi i integrirani izraz
$V=V_0\exp(-\Delta p/K)$; njime provjeravamo pogrešku linearizacije,
a ne valjanost modela stvarnog ulja.

In [ ]:
volumeni_cm3 = np.array([5.0, 10.0, 15.0])
pomaci_mm = np.array([10.0, 20.0, 30.0])
povrsine_mm2 = volumeni_cm3 * 1000 / pomaci_mm
promjeri_mm = np.sqrt(4 * povrsine_mm2 / np.pi)
A = povrsine_mm2[0] * 1e-6
F_blok = .40e6 * A
assert np.allclose(povrsine_mm2, 500.0)
assert np.isclose(F_blok, 200.0)
print('Površine [mm²]:', povrsine_mm2, '; promjeri [mm]:', promjeri_mm)

V0, Vp, dp, K = 250e-6, 5e-6, .40e6, 1e9
Vc = V0 * dp / K
s0, s = Vp/A, (Vp-Vc)/A
rel_pogreska = (s0-s)/s0
Vc_integrirano = -V0 * np.expm1(-dp/K)
assert np.isclose(A*s + Vc, Vp, rtol=1e-12)
assert np.isclose(s*1000, 9.80, rtol=1e-12)
assert rel_pogreska > .01
assert abs(Vc_integrirano/Vc-1) < .00021
print(f'Volumen stlačivanja {Vc*1e6:.3f} cm³; pomak {s*1000:.2f} mm')
print(f'Pogreška pomaka {100*rel_pogreska:.2f} %; kriterij 1 % nije zadovoljen.')

volumeni = np.linspace(0, 500, 101)  # cm³, ista pumpa i isti porast tlaka
greske = (volumeni*1e-6*dp/K)/Vp
assert greske[0] == 0
assert np.all(np.diff(greske) > 0)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
ax.plot(volumeni, 100*greske, label='model stlačivosti')
ax.axhline(1, color='#c62828', ls='--', label='kriterij 1 %')
ax.plot([250], [100*rel_pogreska], 'o', label='Z4')
ax.set(xlabel='početni ukupni volumen tekućine [cm³]', ylabel='pogreška pomaka [%]')
ax.grid(ls=':'); ax.legend(); fig.tight_layout(); plt.show()

## 5. Z5 — dva ograničenja pri izboru pumpe

Radni klip ima 30 cm², opterećenje je 6,0 kN, a radni pomak 10 mm.
Najprije iz opterećenja računamo tlak, zatim za svaki ponuđeni promjer
silu i **zbroj tlačnih hodova**. Povratni potezi nisu u tom zbroju.
Zahtjevi su 150 N i 0,50 m. Provjera rada ne zamjenjuje provjeru granica.

In [ ]:
diametri = np.array([8.0, 9.0, 10.0])
Ap = np.pi*(diametri/1000)**2/4
AL, FL, sL = 30e-4, 6000.0, .010
tlak = FL/AL
sile = tlak*Ap
hodovi = AL*sL/Ap
prihvatljivo = (sile <= 150) & (hodovi <= .50)
assert np.array_equal(prihvatljivo, [False, True, False])
assert np.allclose(sile*hodovi, FL*sL, rtol=1e-12)
assert not np.any((sile <= 120) & (hodovi <= .50))
for d, f, h, ok in zip(diametri, sile, hodovi, prihvatljivo):
    print(f'd={d:.0f} mm: F={f:.3f} N; zbroj hodova={h:.6f} m; prolazi={ok}')

## 6. Z6 — zajamčeni intervali i zajednička odluka

Tri jednaka vođena cilindra imaju po 95 cm². Pumpa promjera 22 mm prima
360 N, a stol se podiže 18 mm. Navedeni intervali **nisu** standardne
nesigurnosti: sve kombinacije faktora sile 0,82–0,90 i volumetrijske
učinkovitosti 0,87–0,93 dopuštene su. Veći faktor sile povećava korisno
opterećenje; manja volumetrijska učinkovitost povećava potreban zbroj hodova.

In [ ]:
Ap6 = np.pi*.022**2/4
p6 = 360/Ap6
G6 = 3*p6*95e-4
sp6 = 3*95e-4*.018/Ap6
etaF = np.array([.82, .86, .90])
etaV = np.array([.87, .90, .93])
korisno = etaF*G6
potrebni_hod = sp6/etaV
assert np.all(korisno >= 22000)
assert np.all(potrebni_hod <= 1.60)
assert np.all(np.diff(korisno) > 0) and np.all(np.diff(potrebni_hod) < 0)
# Omjer korisnog izlaznog i ulaznog rada odgovara produktu obaju faktora.
radni_omjeri = korisno[:, None]*.018/(360*potrebni_hod[None, :])
assert np.allclose(radni_omjeri, etaF[:, None]*etaV[None, :])
print(f'Nominalno: {korisno[1]/1000:.4f} kN i {potrebni_hod[1]:.6f} m')
print(f'Najnepovoljnije: {korisno[0]/1000:.4f} kN i {potrebni_hod[0]:.6f} m')
print(f'Rezerve: {korisno[0]-22000:.2f} N i {1.60-potrebni_hod[0]:.6f} m')

## 7. Protumači i ograniči zaključak

1. Zašto se u Z2 sila povećala 25 puta, a pomak smanjio 25 puta?
2. Zašto jednakost tlaka sama ne provjerava očuvanje rada?
3. Zašto pogreška promjera ulazi u relativnu pogrešku sile s faktorom dva?
4. U Z4 promjena volumena tekućine iznosi oko 0,04 %. Zašto pogreška pomaka iznosi 2 %?
5. Koji bi najveći ukupni volumen u trećem pokusu još zadovoljio kriterij pomaka od 1 %?
6. Zašto podudaranje lineariziranog i integriranog modela ne isključuje zrak ili propuštanje u stvarnom sustavu?
7. Zašto promjer od 8 mm u Z5 prolazi ograničenje sile, ali ipak otpada?
8. Kako se odluka Z6 mijenja ako se potrebna sila povisi na 22,2 kN?
9. Zašto iz provjerenih sila, hodova i nastavnih intervala još ne slijedi dopuštena nosivost stvarnog stroja?